# BioNNE-L: Zero-Shot Hybrid Retrieval with Char TF-IDF

This notebook runs independent RU, EN, and bilingual experiments. For each split, it tunes character TF-IDF late-fusion parameters on dev, reuses dense dev candidates across sparse variants, selects the best dev configuration, and writes test predictions.


In [ ]:
import copy
import json
import logging
import os
from pathlib import Path
import gc

import mlflow
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

from lib.data.text_preprocessing import preprocess_text
from lib.data.vocab_enrichment import (
    enrich_vocab_with_oov_train_dev_terms,
    filter_vocab_for_dataset_language,
    prepare_experiment_vocab,
)
from lib.retrieval.pipelines import make_hybrid_predictions_with_sparse
from lib.retrieval.tuning import evaluate_dev_predictions, grid_search_sparse_params
from lib.utils.logging_utils import configure_logging

In [ ]:
configure_logging(level=logging.INFO, force=True)

ARTIFACTS_DIR = "./artifacts_hybrid_char_tfidf"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

MLFLOW_EXPERIMENT_PREFIX = "bionnel"

DEFAULT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_DEVICE

In [ ]:
LOCAL_MLRUNS_DIR = Path("mlruns").resolve()
mlflow.set_tracking_uri(LOCAL_MLRUNS_DIR.as_uri())
print("MLflow tracking URI:", mlflow.get_tracking_uri())

In [ ]:
## Common Hyperparameters
COMMON_CONFIG = {
    "DEVICE": DEFAULT_DEVICE,  # Device used for dense retrieval inference.
    "USE_TUNING": True,  # Tune sparse/fusion parameters on dev before test inference.
    "ENRICH_VOCABULARY": False,  # Optionally add all unique train/dev mention-CUI pairs beyond the default OOV-only step.
    "DEDUPLICATE_BY_CUI": True,  # Keep at most one candidate per CUI in top-k results.
    "DEV_TOPK": 20,  # Number of dev predictions retained for evaluation.
    "TEST_TOPK": 5,  # Number of final test predictions written per mention.
    "CANDIDATE_POOL_SIZE": 50,  # Dense/sparse candidate pool size before late fusion.
    "FIXED_HYBRID_PARAMS": {
        "TFIDF_MIN_NGRAM": 2,  # Minimum character n-gram length used without tuning.
        "TFIDF_MAX_NGRAM": 2,  # Maximum character n-gram length used without tuning.
        "DENSE_WEIGHT": 0.9,   # Dense score weight used without tuning.
    },
    "HYBRID_PARAM_GRID": {
        "min_ngram": [3],  # Minimum character n-gram values considered during dev tuning.
        "max_ngram": [5],  # Maximum character n-gram values considered during dev tuning.
        "dense_weight": [i / 10 for i in range(11)],  # Dense score weights considered during dev tuning.
    },
    "HYBRID_OPTIMIZE_METRIC": "Acc@1",  # Dev metric used to select tuned fusion parameters.
    "QUERY_BATCH_SIZE": 131_072,  # Number of mentions processed per retrieval batch.
    "DENSE_VOCAB_BATCH_SIZE": 16_384,  # Vocabulary chunk size for dense scoring.
    "SPARSE_VOCAB_BATCH_SIZE": 50000,  # Vocabulary chunk size for sparse scoring.
    "ST_ENCODE_BATCH_SIZE": 8192,  # SentenceTransformer encoding batch size.
}


## Data Loading


In [ ]:
ru_data_train = pd.read_parquet("data/parquet/ru/bionnel_ru_train.parquet")
ru_data_dev = pd.read_parquet("data/parquet/ru/bionnel_ru_dev.parquet")
ru_data_test = pd.read_csv("data/tsv/ru/bionnel_ru_test.tsv", sep="\t")

en_data_train = pd.read_parquet("data/parquet/en/bionnel_en_train.parquet")
en_data_dev = pd.read_parquet("data/parquet/en/bionnel_en_dev.parquet")
en_data_test = pd.read_csv("data/tsv/en/bionnel_en_test.tsv", sep="\t")

bilingual_data_train = pd.read_parquet("data/parquet/bilingual/bionnel_bilingual_train.parquet")
bilingual_data_dev = pd.read_parquet("data/parquet/bilingual/bionnel_bilingual_dev.parquet")
bilingual_data_test = pd.read_csv("data/tsv/bilingual/bionnel_bilingual_test.tsv", sep="\t")

vocab = pd.read_parquet("data/vocabular/bionnel_vocab_bilingual.parquet")

for dataset_df in [
    ru_data_train,
    ru_data_dev,
    ru_data_test,
    en_data_train,
    en_data_dev,
    en_data_test,
    bilingual_data_train,
    bilingual_data_dev,
    bilingual_data_test,
]:
    dataset_df["text"] = dataset_df["text"].map(preprocess_text)

vocab["concept_name"] = vocab["concept_name"].map(preprocess_text)

ALL_ENTITIES_FOR_VOCAB_ENRICHMENT = pd.concat(
    [
        ru_data_train,
        ru_data_dev,
        en_data_train,
        en_data_dev,
        bilingual_data_train,
        bilingual_data_dev,
    ],
    ignore_index=True,
)

vocab = enrich_vocab_with_oov_train_dev_terms(vocab, ALL_ENTITIES_FOR_VOCAB_ENRICHMENT)



print("RU train/dev/test:", ru_data_train.shape, ru_data_dev.shape, ru_data_test.shape)
print("EN train/dev/test:", en_data_train.shape, en_data_dev.shape, en_data_test.shape)
print("Bilingual train/dev/test:", bilingual_data_train.shape, bilingual_data_dev.shape, bilingual_data_test.shape)
print("Vocabulary after default OOV enrichment:", vocab.shape)
print("Train/dev vocab enrichment pool:", ALL_ENTITIES_FOR_VOCAB_ENRICHMENT.shape)
print("Semantic types:", sorted(vocab["semantic_type"].dropna().unique().tolist()))

In [ ]:
## Prediction utils
def evaluate_on_dev(data_df, vocab_df, st_model, cfg, resource_cache=None):
    if resource_cache is None:
        resource_cache = {}
    predictions_df = make_hybrid_predictions_with_sparse(
        entities_df=data_df,
        vocab_df=vocab_df,
        resource_cache=resource_cache,
        st_model=st_model,
        topk=cfg["DEV_TOPK"],
        candidate_pool_size=cfg["CANDIDATE_POOL_SIZE"],
        query_batch_size=cfg["QUERY_BATCH_SIZE"],
        dense_vocab_batch_size=cfg["DENSE_VOCAB_BATCH_SIZE"],
        sparse_query_batch_size=cfg["QUERY_BATCH_SIZE"],
        sparse_vocab_batch_size=cfg["SPARSE_VOCAB_BATCH_SIZE"],
        dense_weight=cfg["DENSE_WEIGHT"],
        sparse_weight=cfg["SPARSE_WEIGHT"],
        sparse_type="char_tfidf",
        sparse_index_params={
            "min_ngram": cfg["TFIDF_MIN_NGRAM"],
            "max_ngram": cfg["TFIDF_MAX_NGRAM"],
            "analyzer": "char",
        },
        st_encode_batch_size=cfg["ST_ENCODE_BATCH_SIZE"],
        deduplicate_by_cui=cfg["DEDUPLICATE_BY_CUI"],
    )
    metrics = evaluate_dev_predictions(predictions_df=predictions_df, data_df=data_df)
    return predictions_df, metrics


def predict_on_test(data_df, vocab_df, st_model, cfg, output_path, resource_cache=None):
    if resource_cache is None:
        resource_cache = {}
    predictions_df = make_hybrid_predictions_with_sparse(
        entities_df=data_df,
        vocab_df=vocab_df,
        resource_cache=resource_cache,
        st_model=st_model,
        topk=cfg["TEST_TOPK"],
        candidate_pool_size=cfg["CANDIDATE_POOL_SIZE"],
        query_batch_size=cfg["QUERY_BATCH_SIZE"],
        dense_vocab_batch_size=cfg["DENSE_VOCAB_BATCH_SIZE"],
        sparse_query_batch_size=cfg["QUERY_BATCH_SIZE"],
        sparse_vocab_batch_size=cfg["SPARSE_VOCAB_BATCH_SIZE"],
        dense_weight=cfg["DENSE_WEIGHT"],
        sparse_weight=cfg["SPARSE_WEIGHT"],
        sparse_type="char_tfidf",
        sparse_index_params={
            "min_ngram": cfg["TFIDF_MIN_NGRAM"],
            "max_ngram": cfg["TFIDF_MAX_NGRAM"],
            "analyzer": "char",
        },
        st_encode_batch_size=cfg["ST_ENCODE_BATCH_SIZE"],
        deduplicate_by_cui=cfg["DEDUPLICATE_BY_CUI"],
    )

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    predictions_df.to_csv(output_path, sep="	", index=False)
    return predictions_df, output_path

In [ ]:
## Config and tuning utils
def build_effective_hybrid_params_from_fixed(cfg):
    fixed = cfg["FIXED_HYBRID_PARAMS"]
    dense_weight = float(fixed["DENSE_WEIGHT"])
    return {
        "TFIDF_MIN_NGRAM": int(fixed["TFIDF_MIN_NGRAM"]),
        "TFIDF_MAX_NGRAM": int(fixed["TFIDF_MAX_NGRAM"]),
        "DENSE_WEIGHT": dense_weight,
        "SPARSE_WEIGHT": float(1.0 - dense_weight),
    }


def tune_hybrid_on_dev(data_df, vocab_df, st_model, cfg):
    tuning_result = grid_search_sparse_params(
        entities_df=data_df,
        vocab_df=vocab_df,
        st_model=st_model,
        sparse_type="char_tfidf",
        sparse_param_grid=cfg["HYBRID_PARAM_GRID"],
        sparse_fixed_index_params={"analyzer": "char"},
        topk=cfg["DEV_TOPK"],
        candidate_pool_size=cfg["CANDIDATE_POOL_SIZE"],
        query_batch_size=cfg["QUERY_BATCH_SIZE"],
        dense_vocab_batch_size=cfg["DENSE_VOCAB_BATCH_SIZE"],
        sparse_query_batch_size=cfg["QUERY_BATCH_SIZE"],
        sparse_vocab_batch_size=cfg["SPARSE_VOCAB_BATCH_SIZE"],
        st_encode_batch_size=cfg["ST_ENCODE_BATCH_SIZE"],
        deduplicate_by_cui=cfg["DEDUPLICATE_BY_CUI"],
        optimize_metric=cfg["HYBRID_OPTIMIZE_METRIC"],
    )

    tuned_cfg = copy.deepcopy(cfg)
    tuned_cfg.update({
        "TFIDF_MIN_NGRAM": int(tuning_result["best_params"]["min_ngram"]),
        "TFIDF_MAX_NGRAM": int(tuning_result["best_params"]["max_ngram"]),
        "DENSE_WEIGHT": float(tuning_result["best_params"]["DENSE_WEIGHT"]),
        "SPARSE_WEIGHT": float(tuning_result["best_params"]["SPARSE_WEIGHT"]),
    })
    return tuning_result, tuned_cfg


def resolve_experiment_config(data_df, vocab_df, st_model, cfg):
    if cfg["USE_TUNING"]:
        return tune_hybrid_on_dev(data_df=data_df, vocab_df=vocab_df, st_model=st_model, cfg=cfg)

    effective_cfg = copy.deepcopy(cfg)
    effective_cfg.update(build_effective_hybrid_params_from_fixed(cfg))
    return None, effective_cfg


def resolve_dev_metrics(data_df, vocab_df, st_model, cfg, tuning_result=None):
    if tuning_result is not None:
        return None, tuning_result["best_metrics"]
    return evaluate_on_dev(data_df=data_df, vocab_df=vocab_df, st_model=st_model, cfg=cfg)

In [ ]:
## MLflow utils
def flatten_config_for_mlflow(prefix, value):
    if isinstance(value, dict):
        flat = {}
        for key, nested_value in value.items():
            nested_prefix = f"{prefix}.{key}" if prefix else str(key)
            flat.update(flatten_config_for_mlflow(nested_prefix, nested_value))
        return flat
    if isinstance(value, (list, tuple)):
        return {prefix: str(list(value))}
    return {prefix: value}


def build_mlflow_params(dataset_name, base_cfg, effective_cfg):
    return {
        "dataset_name": dataset_name,
        "model_name": effective_cfg["MODEL_NAME"],
        "device": effective_cfg["DEVICE"],
        "use_tuning": bool(base_cfg["USE_TUNING"]),
        "enrich_vocabulary": bool(effective_cfg.get("ENRICH_VOCABULARY", False)),
        "deduplicate_by_cui": bool(effective_cfg["DEDUPLICATE_BY_CUI"]),
        "dev_topk": int(effective_cfg["DEV_TOPK"]),
        "test_topk": int(effective_cfg["TEST_TOPK"]),
        "candidate_pool_size": int(effective_cfg["CANDIDATE_POOL_SIZE"]),
        "query_batch_size": int(effective_cfg["QUERY_BATCH_SIZE"]),
        "dense_vocab_batch_size": int(effective_cfg["DENSE_VOCAB_BATCH_SIZE"]),
                "sparse_vocab_batch_size": int(effective_cfg["SPARSE_VOCAB_BATCH_SIZE"]),
        "st_encode_batch_size": int(effective_cfg["ST_ENCODE_BATCH_SIZE"]),
        "hybrid_optimize_metric": effective_cfg["HYBRID_OPTIMIZE_METRIC"],
        "tfidf_min_ngram": int(effective_cfg["TFIDF_MIN_NGRAM"]),
        "tfidf_max_ngram": int(effective_cfg["TFIDF_MAX_NGRAM"]),
        "dense_weight": float(effective_cfg["DENSE_WEIGHT"]),
        "sparse_weight": float(effective_cfg["SPARSE_WEIGHT"]),
    }


def log_run_to_mlflow(dataset_name, base_cfg, effective_cfg, dev_metrics, test_metrics, artifact_paths):
    experiment_name = f"{MLFLOW_EXPERIMENT_PREFIX}-{dataset_name.lower()}"
    mlflow.set_experiment(experiment_name)
    run_name = f"{dataset_name.lower()}-hybrid-char-tfidf-late-fusion"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(build_mlflow_params(dataset_name, base_cfg, effective_cfg))
        mlflow_metrics = {}
        for metric_name, metric_value in dev_metrics.items():
            normalized_name = metric_name.lower().replace("@", "_at_")
            mlflow_metrics[f"dev_{normalized_name}"] = float(metric_value)
        for metric_name, metric_value in test_metrics.items():
            normalized_name = metric_name.lower().replace("@", "_at_")
            mlflow_metrics[f"test_{normalized_name}"] = float(metric_value)
        mlflow.log_metrics(mlflow_metrics)

        artifact_dir_map = {
            "test_predictions": "predictions",
            "test_metrics": "metrics",
            "metrics_summary": "metrics",
            "base_config": "configs",
            "effective_config": "configs",
            "config": "configs",
            "tuning_results": "tuning",
        }
        for artifact_name, artifact_path in artifact_paths.items():
            mlflow.log_artifact(artifact_path, artifact_path=artifact_dir_map.get(artifact_name))
        return mlflow.active_run().info.run_id


In [ ]:
## Artifacts utils
def get_artifact_dir(dataset_name):
    artifact_dir = Path(ARTIFACTS_DIR) / dataset_name.lower()
    artifact_dir.mkdir(parents=True, exist_ok=True)
    return artifact_dir


def save_local_artifacts(dataset_name, base_cfg, effective_cfg, dev_metrics, test_predictions_df, test_metrics, test_predictions_path=None, tuning_result=None):
    artifact_dir = get_artifact_dir(dataset_name)

    if test_predictions_path is None:
        test_predictions_path = artifact_dir / "test_predictions.tsv"
    else:
        test_predictions_path = Path(test_predictions_path)
    test_metrics_path = artifact_dir / "test_metrics.json"
    metrics_table_path = artifact_dir / "metrics_summary.tsv"
    base_config_path = artifact_dir / "base_config.json"
    effective_config_path = artifact_dir / "effective_config.json"

    if not test_predictions_path.exists():
        test_predictions_df.to_csv(test_predictions_path, sep="\t", index=False)
    test_metrics_path.write_text(json.dumps(test_metrics, indent=2, ensure_ascii=False))
    pd.DataFrame([
        {"split": "dev", **dev_metrics},
        {"split": "test", **test_metrics},
    ]).to_csv(metrics_table_path, sep="\t", index=False)
    base_config_path.write_text(json.dumps(flatten_config_for_mlflow("", base_cfg), indent=2, ensure_ascii=False))
    effective_config_path.write_text(json.dumps(flatten_config_for_mlflow("", effective_cfg), indent=2, ensure_ascii=False))

    artifact_paths = {
        "test_predictions": str(test_predictions_path),
        "test_metrics": str(test_metrics_path),
        "metrics_summary": str(metrics_table_path),
        "base_config": str(base_config_path),
        "effective_config": str(effective_config_path),
    }

    if tuning_result is not None:
        tuning_results_path = artifact_dir / "tuning_results.csv"
        tuning_result["results_df"].to_csv(tuning_results_path, index=False)
        artifact_paths["tuning_results"] = str(tuning_results_path)

    return artifact_paths

## RU Experiment

In [ ]:
RU_CONFIG = copy.deepcopy(COMMON_CONFIG)
RU_CONFIG["MODEL_NAME"] = "andorei/BERGAMOT-multilingual-GAT"

In [ ]:
ru_st_model = SentenceTransformer(RU_CONFIG["MODEL_NAME"], device=RU_CONFIG["DEVICE"])

In [ ]:
ru_vocab = prepare_experiment_vocab(vocab, ALL_ENTITIES_FOR_VOCAB_ENRICHMENT, RU_CONFIG)

ru_tuning_result, RU_TUNED_CONFIG = resolve_experiment_config(
    data_df=ru_data_dev,
    vocab_df=ru_vocab,
    st_model=ru_st_model,
    cfg=RU_CONFIG,
)

In [ ]:
ru_dev_predictions_df, ru_dev_metrics = resolve_dev_metrics(
    data_df=ru_data_dev,
    vocab_df=ru_vocab,
    st_model=ru_st_model,
    cfg=RU_TUNED_CONFIG,
    tuning_result=ru_tuning_result,
)

In [ ]:
ru_tuning_result["results_df"] if ru_tuning_result is not None else "Tuning disabled"

In [ ]:
RU_TUNED_CONFIG, ru_dev_metrics

In [ ]:
ru_test_resource_cache = {}
ru_artifact_dir = get_artifact_dir("ru")
ru_test_predictions_df, ru_test_predictions_path = predict_on_test(
    data_df=ru_data_test,
    vocab_df=ru_vocab,
    st_model=ru_st_model,
    cfg=RU_TUNED_CONFIG,
    output_path=ru_artifact_dir / "test_predictions.tsv",
    resource_cache=ru_test_resource_cache,
)

ru_test_metrics = evaluate_dev_predictions(
    predictions_df=ru_test_predictions_df,
    data_df=ru_data_test,
)


In [ ]:
ru_artifact_paths = save_local_artifacts(
    dataset_name="ru",
    base_cfg=RU_CONFIG,
    effective_cfg=RU_TUNED_CONFIG,
    dev_metrics=ru_dev_metrics,
    test_predictions_df=ru_test_predictions_df,
    test_metrics=ru_test_metrics,
    test_predictions_path=ru_test_predictions_path,
    tuning_result=ru_tuning_result,
)

In [ ]:
ru_mlflow_run_id = log_run_to_mlflow(
    dataset_name="ru",
    base_cfg=RU_CONFIG,
    effective_cfg=RU_TUNED_CONFIG,
    dev_metrics=ru_dev_metrics,
    test_metrics=ru_test_metrics,
    artifact_paths=ru_artifact_paths,
)

ru_artifact_paths, ru_mlflow_run_id

In [ ]:
del ru_st_model
gc.collect()
torch.cuda.empty_cache()

## EN Experiment


In [ ]:
EN_CONFIG = copy.deepcopy(COMMON_CONFIG)
EN_CONFIG["MODEL_NAME"] = "andorei/gebert_eng_gat"

In [ ]:
en_st_model = SentenceTransformer(EN_CONFIG["MODEL_NAME"], device=EN_CONFIG["DEVICE"])

In [ ]:
en_vocab = prepare_experiment_vocab(
    filter_vocab_for_dataset_language(vocab, "en"),
    ALL_ENTITIES_FOR_VOCAB_ENRICHMENT,
    EN_CONFIG,
)

en_tuning_result, EN_TUNED_CONFIG = resolve_experiment_config(
    data_df=en_data_dev,
    vocab_df=en_vocab,
    st_model=en_st_model,
    cfg=EN_CONFIG,
)

In [ ]:
en_dev_predictions_df, en_dev_metrics = resolve_dev_metrics(
    data_df=en_data_dev,
    vocab_df=en_vocab,
    st_model=en_st_model,
    cfg=EN_TUNED_CONFIG,
    tuning_result=en_tuning_result,
)

In [ ]:
en_tuning_result["results_df"] if en_tuning_result is not None else "Tuning disabled"

In [ ]:
EN_TUNED_CONFIG, en_dev_metrics

In [ ]:
en_test_resource_cache = {}
en_artifact_dir = get_artifact_dir("en")
en_test_predictions_df, en_test_predictions_path = predict_on_test(
    data_df=en_data_test,
    vocab_df=en_vocab,
    st_model=en_st_model,
    cfg=EN_TUNED_CONFIG,
    output_path=en_artifact_dir / "test_predictions.tsv",
    resource_cache=en_test_resource_cache,
)

en_test_metrics = evaluate_dev_predictions(
    predictions_df=en_test_predictions_df,
    data_df=en_data_test,
)


In [ ]:
en_artifact_paths = save_local_artifacts(
    dataset_name="en",
    base_cfg=EN_CONFIG,
    effective_cfg=EN_TUNED_CONFIG,
    dev_metrics=en_dev_metrics,
    test_predictions_df=en_test_predictions_df,
    test_metrics=en_test_metrics,
    test_predictions_path=en_test_predictions_path,
    tuning_result=en_tuning_result,
)

In [ ]:
en_mlflow_run_id = log_run_to_mlflow(
    dataset_name="en",
    base_cfg=EN_CONFIG,
    effective_cfg=EN_TUNED_CONFIG,
    dev_metrics=en_dev_metrics,
    test_metrics=en_test_metrics,
    artifact_paths=en_artifact_paths,
)

en_artifact_paths, en_mlflow_run_id

In [ ]:
del en_st_model
gc.collect()
torch.cuda.empty_cache()

## Bilingual Experiment


In [ ]:
BILINGUAL_CONFIG = copy.deepcopy(COMMON_CONFIG)
BILINGUAL_CONFIG["MODEL_NAME"] = "andorei/BERGAMOT-multilingual-GAT"

In [ ]:
bilingual_st_model = SentenceTransformer(BILINGUAL_CONFIG["MODEL_NAME"], device=BILINGUAL_CONFIG["DEVICE"])

In [ ]:
bilingual_vocab = prepare_experiment_vocab(vocab, ALL_ENTITIES_FOR_VOCAB_ENRICHMENT, BILINGUAL_CONFIG)

bilingual_tuning_result, BILINGUAL_TUNED_CONFIG = resolve_experiment_config(
    data_df=bilingual_data_dev,
    vocab_df=bilingual_vocab,
    st_model=bilingual_st_model,
    cfg=BILINGUAL_CONFIG,
)

In [ ]:
bilingual_dev_predictions_df, bilingual_dev_metrics = resolve_dev_metrics(
    data_df=bilingual_data_dev,
    vocab_df=bilingual_vocab,
    st_model=bilingual_st_model,
    cfg=BILINGUAL_TUNED_CONFIG,
    tuning_result=bilingual_tuning_result,
)

In [ ]:
bilingual_tuning_result["results_df"] if bilingual_tuning_result is not None else "Tuning disabled"

In [ ]:
BILINGUAL_TUNED_CONFIG, bilingual_dev_metrics

In [ ]:
bilingual_test_resource_cache = {}
bilingual_artifact_dir = get_artifact_dir("bilingual")
bilingual_test_predictions_df, bilingual_test_predictions_path = predict_on_test(
    data_df=bilingual_data_test,
    vocab_df=bilingual_vocab,
    st_model=bilingual_st_model,
    cfg=BILINGUAL_TUNED_CONFIG,
    output_path=bilingual_artifact_dir / "test_predictions.tsv",
    resource_cache=bilingual_test_resource_cache,
)

bilingual_test_metrics = evaluate_dev_predictions(
    predictions_df=bilingual_test_predictions_df,
    data_df=bilingual_data_test,
)


In [ ]:
bilingual_artifact_paths = save_local_artifacts(
    dataset_name="bilingual",
    base_cfg=BILINGUAL_CONFIG,
    effective_cfg=BILINGUAL_TUNED_CONFIG,
    dev_metrics=bilingual_dev_metrics,
    test_predictions_df=bilingual_test_predictions_df,
    test_metrics=bilingual_test_metrics,
    test_predictions_path=bilingual_test_predictions_path,
    tuning_result=bilingual_tuning_result,
)

In [ ]:
bilingual_mlflow_run_id = log_run_to_mlflow(
    dataset_name="bilingual",
    base_cfg=BILINGUAL_CONFIG,
    effective_cfg=BILINGUAL_TUNED_CONFIG,
    dev_metrics=bilingual_dev_metrics,
    test_metrics=bilingual_test_metrics,
    artifact_paths=bilingual_artifact_paths,
)

bilingual_artifact_paths, bilingual_mlflow_run_id

In [ ]:
del bilingual_st_model
gc.collect()
torch.cuda.empty_cache()